# AlexNet（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，AlexNetを用いてCIFAR100データセットの100クラス物体認識を行い，ReLUやDropout，Overlapping Max Poolingなど，現在の深層学習の基礎となった技術について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100はCIFAR10と同じく32×32ピクセルのカラー画像データセットですが，クラス数が100（1クラスあたり学習用500枚，評価用100枚）とより細かいクラス分けになっています．

学習用データに対しては，`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## AlexNet
AlexNetは，2012年のILSVRC（ImageNet Large Scale Visual Recognition Challenge）においてそれまでの手法を大きく上回る精度で優勝したモデルであり，畳み込みニューラルネットワークによる大規模画像認識の有効性を世に知らしめた最初のモデルです．

AlexNetは5層の畳み込み層と3層の全結合層から構成される比較的シンプルな構造ですが，以下のような当時としては先進的な工夫が盛り込まれており，これらは現在の深層学習においても広く使われている基本技術となっています．

- **ReLU活性化関数**: それまで主流だったsigmoidやtanhに比べて勾配消失が起きにくく，学習を大幅に高速化できる．
- **Dropout**: 全結合層の各ユニットを学習時に確率的に無効化することで，パラメータ数の多い全結合層の過学習を抑制する．
- **Overlapping Max Pooling**: プーリング領域を一部重複させながらダウンサンプリングすることで，通常のプーリングよりわずかに高い精度が得られる．

なお，原論文ではこれに加えてLocal Response Normalization（LRN）という正規化層も用いられていますが，後年登場したBatch Normalizationに比べて効果が薄いことが分かっており，現在ではほとんど使われないため本ノートブックでは省略します．

### CIFAR100への入力サイズの適応
原論文のAlexNetは，224×224ピクセルのImageNet画像を入力として設計されており，1層目の畳み込みに11×11・ストライド4という大きなカーネルを用いることで，一気に空間サイズを縮小します．しかし，CIFAR100の入力は32×32ピクセルと非常に小さいため，このカーネルサイズ・ストライドをそのまま適用すると特徴マップのサイズが1〜2層目で0以下になってしまい，ネットワークを構成できません．そこで本ノートブックでは，1・2層目の畳み込みのカーネルサイズを5×5，ストライドを1に変更し，緩やかに空間サイズを縮小するように調整しています（詳細は本ノートブック末尾の「ImageNet版AlexNetとCIFAR版AlexNetの違い」を参照）．

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, n_class=100):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),  # 32 -> 16

            nn.Conv2d(64, 192, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),  # 16 -> 8

            nn.Conv2d(192, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),  # 8 -> 4
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, n_class),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

## ネットワークの作成
定義したネットワークを作成します．`.to(device)`によりモデルのパラメータを`device`上に配置します．

最適化手法にはモーメンタムSGDを用い，学習率0.01，モーメンタム0.9，weight decayを5e-4とします．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
model = AlexNet(n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版AlexNetとCIFAR版AlexNetの違い
このノートブックで実装したAlexNetは，32×32ピクセルのCIFAR100に合わせて構造を変更したものです．原論文のImageNet版AlexNetとは，主に次の点が異なります．

| | ImageNet版（原論文） | CIFAR版（本ノートブック） |
|---|---|---|
| 入力画像サイズ | 224×224 | 32×32 |
| conv1のカーネルサイズ / ストライド | 11×11 / stride 4 | 5×5 / stride 1 |
| conv2のカーネルサイズ | 5×5 | 5×5（変更なし） |
| プーリング | Overlapping MaxPool（3×3, stride 2）×3回 | 同様に3回（各層の入力サイズが小さいため縮小幅は緩やか） |
| Local Response Normalization | あり | なし（Batch Normalization等に比べ効果が薄いため省略） |
| 全結合層のユニット数 | 4096-4096-1000 | 1024-1024-100 |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

このように，AlexNetは元々224×224という大きな入力を前提に設計されているため，1・2層目の畳み込みのカーネルサイズとストライドを緩めることが，CIFAR100のような小さな入力画像に適用する上で最も大きな変更点となります．

## 課題

1. Dropoutを取り除いたネットワークを作成し，学習曲線やテスト精度がどのように変化するか（過学習のしやすさ）を比較しましょう．

2. conv1・conv2のカーネルサイズやストライドを変更し，パラメータ数や認識精度への影響を確認しましょう．

3. 全結合層のユニット数（1024など）を変更し，パラメータ数と認識精度の関係を確認しましょう．